In [1]:
import pandas as pd
import numpy as np
import joblib
import shap
import re

from sklearn.metrics import accuracy_score

In [4]:
rf_model = joblib.load( "models/rf_model.pkl")
feature_names = joblib.load( "models/feature_names.pkl")

boss = pd.read_csv("datasets/final_merged.csv")
region_profiles = pd.read_csv("datasets/employee_cleaned.csv")

boss["employee_id"] = region_profiles["employee_id"]

print("Model Loaded Successfully!")
print("Employees :", len(boss))

Model Loaded Successfully!
Employees : 10000


In [175]:

def get_employee(employee_id):
    

    employee = boss[boss["employee_id"] == employee_id]

    if employee.empty:
        return None

    return employee.iloc[0]


def prepare_features(employee_row):
   

    X = employee_row[feature_names]

    return pd.DataFrame([X])


def predict_probability(employee_row):

    X = prepare_features(employee_row)

    probability = rf_model.predict_proba(X)[0][1]

    prediction = rf_model.predict(X)[0]

    return prediction, probability

In [176]:


def predict_enrollment(employee_id):
    

    employee = get_employee(employee_id)

    if employee is None:
        return f" Employee ID {employee_id} not found."

    prediction, probability = predict_probability(employee)

  
    print("ENROLLMENT PREDICTION")
    

    print(f"Employee ID : {employee_id}")

    if prediction == 1:
        print("Prediction  : Likely to Enroll")
    else:
        print("Prediction  : Unlikely to Enroll")

    print(f"Probability : {probability:.2%}")

    return prediction, probability

In [177]:

def rank_outreach_candidates(region_name):
   

    region_df = boss[
        boss["region"].astype(str).str.lower() == region_name.lower()
    ].copy()

    if len(region_df) == 0:
        print("Region not found.")
        return

   
    capacity = int(region_df["hr_outreach_capacity"].iloc[0])

    
    X_region = region_df[feature_names]

    probabilities = rf_model.predict_proba(X_region)[:, 1]

    region_df["Enrollment Probability"] = probabilities

  
    ranked = region_df.sort_values(
        by="Enrollment Probability",
        ascending=False
    )

    top_k = ranked.head(capacity)

   
    print("TOP OUTREACH CANDIDATES")


    print(f"Region               : {region_name}")
    print(f"HR Outreach Capacity : {capacity}")
    print(f"Employees in Region  : {len(region_df)}")

    print("\nTop Candidates:\n")

    display(
        top_k[
            [
                "employee_id",
                "Enrollment Probability",
                "employment_type",
                "salary",
                "has_dependents",
                "prior_year_enrolled"
            ]
        ]
    )

    return top_k

    return top_k

In [178]:



FORBIDDEN_FEATURES = [
    "legacy_propensity_score",
    "outreach_notes"
]


SENSITIVE_FEATURES = [
    "age",
    "gender",
    "marital_status"
]


ENROLLMENT_THRESHOLD = 0.50

In [179]:
REGION_MAP = {
    "midwest": 0,
    "northeast": 1,
    "south": 2,
    "west": 3
}

def rank_outreach_candidates(region_name):

    region_name = region_name.strip().lower()

    if region_name not in REGION_MAP:
        print("Invalid region.")
        return

    region_code = REGION_MAP[region_name]

    region_df = boss[boss["region"] == region_code].copy()

    if region_df.empty:
        print("Region not found.")
        return

    capacity = int(region_df["hr_outreach_capacity"].iloc[0])

    X_region = region_df[feature_names]

    region_df["Enrollment Probability"] = rf_model.predict_proba(X_region)[:, 1]

    
    region_df = region_df[
        region_df["Enrollment Probability"] >= ENROLLMENT_THRESHOLD
    ]

    ranked = region_df.sort_values(
        by="Enrollment Probability",
        ascending=False
    )

    top_k = ranked.head(capacity)

  
    print("TOP OUTREACH CANDIDATES")


    print(f"Region               : {region_name.title()}")
    print(f"HR Outreach Capacity : {capacity}")
    print(f"Eligible Employees   : {len(region_df)}")

    display(
        top_k[
            [
                "employee_id",
                "Enrollment Probability",
                "employment_type",
                "salary",
                "has_dependents",
                "prior_year_enrolled"
            ]
        ]
    )

    return top_k

In [180]:
REGION_MAP = {
    "midwest": 0,
    "northeast": 1,
    "south": 2,
    "west": 3
}

def lookup_region_profile(region_name):

    region_name = region_name.strip().lower()

    if region_name not in REGION_MAP:
        print(" Invalid Region")
        return

    region_code = REGION_MAP[region_name]

    region_df = boss[boss["region"] == region_code]

    if region_df.empty:
        print("Region not found.")
        return

    profile = {
        "Region": region_name.title(),
        "Employees": len(region_df),
       # "Historical Enrollment Rate":
          #  round(region_df["hist_enrollment_rate_region"].iloc[0] * 100, 2),

        "Average Salary":
            round(region_df["avg_salary_region"].iloc[0], 2),

        "Average Premium":
            round(region_df["avg_premium_cost_usd"].iloc[0], 2),

        "Broker Rating":
            round(region_df["benefits_broker_rating"].iloc[0], 2),

        "HR Outreach Capacity":
            int(region_df["hr_outreach_capacity"].iloc[0]),

        "Open Enrollment Window":
            int(region_df["open_enrollment_window_days"].iloc[0]),

        "State Mandate Level":
            int(region_df["state_mandate_level"].iloc[0])
    }

   
    print("REGION PROFILE")
    

    for k, v in profile.items():
        print(f"{k:<30}: {v}")

    return profile

In [181]:

X_background = boss[feature_names].sample(
    min(200, len(boss)),
    random_state=42
)

explainer = shap.TreeExplainer(
    rf_model,
    X_background
)


In [182]:

BLOCKED_FEATURES = [
    "age",
    "gender",
    "marital_status",
    "legacy_propensity_score"
]


def explain_prediction(employee_id):

    employee = get_employee(employee_id)

    if employee is None:
        print("Employee not found.")
        return


    prediction, probability = predict_probability(employee)


    X = prepare_features(employee)


    shap_values = explainer.shap_values(X)

    values = np.array(shap_values)



    if values.ndim == 3:
       
        values = values[0,:,1]

    elif values.ndim == 2:
        values = values[0]

    elif isinstance(shap_values, list):
       
        values = shap_values[1][0]


    values = values.flatten()

    shap_df = pd.DataFrame({

        "feature": feature_names,

        "impact": values

    })


    

    shap_df = shap_df[
        ~shap_df["feature"].isin(
            BLOCKED_FEATURES
        )
    ]


    shap_df["abs_impact"] = (
        shap_df["impact"]
        .abs()
    )


    top_features = (
        shap_df
        .sort_values(
            "abs_impact",
            ascending=False
        )
        .head(5)
    )


  

   
    print("PREDICTION EXPLANATION")
    

    print(f"Employee ID : {employee_id}")

    if prediction == 1:
        print("Prediction  : Likely to Enroll")
    else:
        print("Prediction  : Unlikely to Enroll")


    print(
        f"Probability : {probability:.2%}"
    )


    print("\nMain Factors:")


    for _, row in top_features.iterrows():

        direction = (
            "increased"
            if row["impact"] > 0
            else "decreased"
        )

        print(
            f"- {row['feature']} "
            f"{direction} enrollment likelihood"
        )


    return top_features

In [183]:
explain_prediction(19434)

PREDICTION EXPLANATION
Employee ID : 19434
Prediction  : Unlikely to Enroll
Probability : 37.46%

Main Factors:
- employment_type decreased enrollment likelihood
- has_dependents increased enrollment likelihood
- salary increased enrollment likelihood
- salary_diff_region increased enrollment likelihood
- prior_year_enrolled decreased enrollment likelihood


,feature,impact,abs_impact
4,employment_type,-0.214027,0.214027
6,has_dependents,0.156986,0.156986
3,salary,0.084993,0.084993
25,salary_diff_region,0.058629,0.058629
11,prior_year_enrolled,-0.030019,0.030019


In [184]:


def validate_raw_row(row):

    issues = []


    
    missing = row[
        row.isna()
    ].index.tolist()

    if missing:
        issues.append(
            f"Missing values: {missing}"
        )


    
    if "salary" in row:

        if row["salary"] <= 0:
            issues.append(
                "Salary must be positive"
            )


   
    if "tenure_years" in row:

        if row["tenure_years"] < 0:
            issues.append(
                "Tenure cannot be negative"
            )


    
    if (
        "age" in row and
        "tenure_years" in row
    ):

        if row["tenure_years"] > row["age"] - 18:

            issues.append(
                "Tenure exceeds possible working years"
            )


   
    missing_features = [
        col for col in feature_names
        if col not in row.index
    ]

    if missing_features:
        issues.append(
            f"Missing model features: {missing_features}"
        )


   
    print("RAW ROW VALIDATION")
   


    if not issues:

        print(" Row looks valid")

    else:

        for issue in issues:
            print( issue)


    return issues

In [185]:



FORBIDDEN_FIELDS = [
    "legacy_propensity_score"
]


def check_forbidden_request(query):

    query = query.lower()

    for field in FORBIDDEN_FIELDS:

        if field in query:

            return (
                "I cannot use legacy_propensity_score "
                "because it is a restricted feature "
                "that may reconstruct the target variable."
            )

    return None

In [186]:
def compare_model_vs_legacy(employee_id):

    raw_row = boss_audit[
        boss_audit["employee_id"] == employee_id
    ].iloc[0]


    model_row = boss[
        boss["employee_id"] == employee_id
    ].iloc[0]


    X = model_row[feature_names].values.reshape(1,-1)


    model_probability = rf_model.predict_proba(X)[0][1]


    legacy_score = region_profiles["legacy_propensity_score"]


    print(f"Employee ID: {employee_id}")
    print(f"RF Probability: {model_probability:.2%}")
    print(f"Legacy Score: {legacy_score:.2%}")
    print(
        f"Difference: {abs(model_probability-legacy_score):.2%}"
    )

In [187]:


def agent(query):

    query_lower = query.lower()



    refusal = check_forbidden_request(query)

    if refusal:
        print(refusal)
        return



   

    ranking_keywords = [
        "top",
        "priority",
        "priorities",
        "outreach",
        "contact",
        "candidates"
    ]

    if any(word in query_lower for word in ranking_keywords):

        for region in REGION_MAP:

            if region in query_lower:

                return rank_outreach_candidates(
                    region
                )




    if (
        "why" in query_lower
        or "explain" in query_lower
    ):

        numbers = [
            int(word)
            for word in query.split()
            if word.isdigit()
        ]

        if numbers:

            return explain_prediction(
                numbers[0]
            )




    if (
        "profile" in query_lower
        or "stats" in query_lower
        or "region" in query_lower
    ):

        for region in REGION_MAP:

            if region in query_lower:

                return lookup_region_profile(
                    region
                )




    if (
        "wrong" in query_lower
        or "issue" in query_lower
        or "problem" in query_lower
    ):

        print(
            "Please provide the employee row to validate."
        )

        return



    print(
        "I could not understand the request."
    )

In [188]:
agent(
"who are the top 20 outreach priorities in the Midwest this window?"
)

TOP OUTREACH CANDIDATES
Region               : Midwest
HR Outreach Capacity : 469
Eligible Employees   : 1650


,employee_id,Enrollment Probability,employment_type,salary,has_dependents,prior_year_enrolled
9947,16098,0.994583,1,106880.90,1,1
2607,15808,0.994583,1,99496.67,1,1
6451,13681,0.994583,1,96036.33,1,1
6502,19169,0.994583,1,98852.22,1,1
3817,11353,0.994315,1,97415.96,1,1
...,...,...,...,...,...,...
347,15730,0.980047,1,70986.78,1,0
8183,19114,0.980047,1,66899.89,1,0
8981,11308,0.980047,1,75163.14,1,0
7342,10072,0.979957,1,96315.81,1,0


,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,last_contact_channel,...,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level,premium_salary_percentage,outreach_capacity_ratio,salary_diff_region,employee_id,Enrollment Probability
9947,63,0,1,106880.90,1,0,1,4.5,1,0,...,595,4.4,469,18,0,0.556694,0.188505,41959.44,16098,0.994583
2607,64,0,2,99496.67,1,0,1,7.7,1,0,...,595,4.4,469,18,0,0.598010,0.188505,34575.21,15808,0.994583
6451,62,1,1,96036.33,1,0,1,2.2,1,0,...,595,4.4,469,18,0,0.619557,0.188505,31114.87,13681,0.994583
6502,42,1,2,98852.22,1,0,1,2.5,1,0,...,595,4.4,469,18,0,0.601909,0.188505,33930.76,19169,0.994583
3817,50,0,1,97415.96,1,0,1,1.1,1,0,...,595,4.4,469,18,0,0.610783,0.188505,32494.50,11353,0.994315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,63,0,1,70986.78,1,0,1,10.9,1,0,...,595,4.4,469,18,0,0.838184,0.188505,6065.32,15730,0.980047
8183,50,0,1,66899.89,1,0,1,8.4,1,2,...,595,4.4,469,18,0,0.889389,0.188505,1978.43,19114,0.980047
8981,59,1,1,75163.14,1,0,1,6.3,1,0,...,595,4.4,469,18,0,0.791611,0.188505,10241.68,11308,0.980047
7342,39,1,2,96315.81,1,0,1,1.6,1,0,...,595,4.4,469,18,0,0.617759,0.188505,31394.35,10072,0.979957


In [189]:
agent(
"what is the Midwest region profile?"
)

REGION PROFILE
Region                        : Midwest
Employees                     : 2488
Average Salary                : 64921.46
Average Premium               : 595
Broker Rating                 : 4.4
HR Outreach Capacity          : 469
Open Enrollment Window        : 18
State Mandate Level           : 0


{'Region': 'Midwest',
 'Employees': 2488,
 'Average Salary': np.float64(64921.46),
 'Average Premium': np.int64(595),
 'Broker Rating': np.float64(4.4),
 'HR Outreach Capacity': 469,
 'Open Enrollment Window': 18,
 'State Mandate Level': 0}

In [190]:
agent(
"why is employee 11234 predicted to enroll?"
)

PREDICTION EXPLANATION
Employee ID : 11234
Prediction  : Likely to Enroll
Probability : 66.25%

Main Factors:
- has_dependents decreased enrollment likelihood
- salary increased enrollment likelihood
- salary_diff_region increased enrollment likelihood
- prior_year_enrolled decreased enrollment likelihood
- employment_type decreased enrollment likelihood


,feature,impact,abs_impact
6,has_dependents,-0.182001,0.182001
3,salary,0.101388,0.101388
25,salary_diff_region,0.077112,0.077112
11,prior_year_enrolled,-0.018981,0.018981
4,employment_type,-0.004331,0.004331


In [191]:
agent(
    "show me the South region profile"
)

REGION PROFILE
Region                        : South
Employees                     : 2424
Average Salary                : 65200.49
Average Premium               : 481
Broker Rating                 : 3.3
HR Outreach Capacity          : 324
Open Enrollment Window        : 17
State Mandate Level           : 2


{'Region': 'South',
 'Employees': 2424,
 'Average Salary': np.float64(65200.49),
 'Average Premium': np.int64(481),
 'Broker Rating': np.float64(3.3),
 'HR Outreach Capacity': 324,
 'Open Enrollment Window': 17,
 'State Mandate Level': 2}

In [192]:
agent(
    "Why is employee 17825 predicted to enroll?"
)

PREDICTION EXPLANATION
Employee ID : 17825
Prediction  : Likely to Enroll
Probability : 75.00%

Main Factors:
- has_dependents increased enrollment likelihood
- salary increased enrollment likelihood
- salary_diff_region increased enrollment likelihood
- employment_type increased enrollment likelihood
- prior_year_enrolled decreased enrollment likelihood


,feature,impact,abs_impact
6,has_dependents,0.158509,0.158509
3,salary,0.103983,0.103983
25,salary_diff_region,0.068312,0.068312
4,employment_type,0.064732,0.064732
11,prior_year_enrolled,-0.022457,0.022457
